### 1. Shared setup (questions, contexts, reference)

In [ ]:
import pandas as pd

questions = [
    "Which part of the brain does short-term memory seem to rely on?",
    "What provided the Roman senate with exuberance?",
    "What area did the Hasan-jalalians command?",
]

contexts = [
    "Short-term memory is supported by transient patterns of neural activity in the frontal lobe (especially the dorsolateral prefrontal cortex) and the parietal lobe.",
    "In 62 BC, Pompey returned victorious from Asia. The Senate, elated by its successes against Catiline, celebrated.",
    "The noble family of Orbelians... The Armenian family of Hasan-Jalalians controlled the provinces of Artsakh and Utik.",
]

references = [
    "frontal lobe and the parietal lobe",
    "Successes against Catiline.",
    "The provinces of Artsakh and Utik.",
]

### 2. Call Gemini via Vertex AI (Python GenAI SDK)
This uses the new google-genai SDK for Gemini on Vertex AI.

In [ ]:
from google import genai
from google.genai.types import HttpOptions

# Define project and location for Vertex AI
PROJECT_ID = "eduardo-calendly-playground" # TODO: Replace with your actual project ID
LOCATION = "us-central1"

# Authenticate user for Google Cloud services
from google.colab import auth
auth.authenticate_user()

client_gemini = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)

def call_gemini(prompt: str) -> str:
    resp = client_gemini.models.generate_content(
        model="gemini-2.5-flash",   # or gemini-2.0-pro if you prefer
        contents=prompt,
    )
    return resp.text.strip()

gemini_responses = []
for q, c in zip(questions, contexts):
    prompt = f"""You are a concise QA assistant.
Question: {q}
Context: {c}
Answer in one short phrase, based only on the context."""
    gemini_responses.append(call_gemini(prompt))

### 3. Call OpenAI directly (public OpenAI API)
This uses the official openai Python library against OpenAI’s own endpoint.

In [ ]:
from openai import OpenAI

client_openai = OpenAI(api_key="sk-proj-CUSj")

def call_openai(prompt: str) -> str:
    resp = client_openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a concise QA assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()

openai_responses = []
for q, c in zip(questions, contexts):
    prompt = f"""Answer in one short phrase, based only on this context.
Question: {q}
Context: {c}"""
    openai_responses.append(call_openai(prompt))

### 4. Build eval dataset for Vertex AI
You now have both sets of responses; construct two SQuAD‑style DataFrames for evaluation

In [ ]:
dual_df = pd.DataFrame({
    "prompt": [
        f"Question: {q}\nContext: {c}"
        for q, c in zip(questions, contexts)
    ],
    "reference": references,
    "gemini_response": gemini_responses,
    "openai_response": openai_responses,
})

gemini_eval_df = dual_df[["prompt", "gemini_response", "reference"]].rename(
    columns={"gemini_response": "response"}
)
openai_eval_df = dual_df[["prompt", "openai_response", "reference"]].rename(
    columns={"openai_response": "response"}
)

### 5. Run Vertex AI eval for both models
You can now plug each dataset into Vertex AI’s Gen AI Evaluation Service with metrics like ROUGE and exact_match, plus model‑based metrics with Gemini as judge.

In [ ]:
import pandas as pd
!pip install google-cloud-aiplatform
from vertexai import init as vertexai_init
from vertexai.evaluation import EvalTask

vertexai_init(project=PROJECT_ID, location=LOCATION)


metrics = [
    "question_answering_quality",  # model-based, Gemini judge
    "groundedness",                # model-based
    "rouge",                       # computation-based
    "exact_match",                 # computation-based
]

gemini_task = EvalTask(
    dataset=gemini_eval_df,
    metrics=metrics,
    experiment="rag-compare-gemini-openai",
)
gemini_results = gemini_task.evaluate()

openai_task = EvalTask(
    dataset=openai_eval_df,
    metrics=metrics,
    experiment="rag-compare-gemini-openai",
)
openai_results = openai_task.evaluate()

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:04<00:00,  2.83it/s]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:4.25744905900001 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
 92%|█████████▏| 11/12 [00:05<00:00,  1.11it/s]

### 6. Visualize Gemini vs OpenAI
Use notebook_utils to see the difference per metric

In [ ]:
from vertexai.preview.evaluation import notebook_utils

results = [
    ("Gemini 2.5 Flash", gemini_results),
    ("OpenAI GPT-4o", openai_results),
]

notebook_utils.display_radar_plot(
    results,
    metrics=["question_answering_quality", "groundedness"]
)

notebook_utils.display_bar_plot(
    results,
    metrics=["rouge", "exact_match"]
)
